# 09 — Visualization Dashboard

Comprehensive visualization of model results, class distributions, feature importance,
confusion matrices, ROC curves, detection timeline, and severity breakdown.
Provides an at-a-glance view of system performance.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, roc_curve, auc, classification_report
)
from sklearn.preprocessing import label_binarize
import joblib
import os
import sys
import time
from datetime import datetime, timedelta

sys.path.insert(0, os.path.join(os.path.dirname("__file__"), ".."))

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 11

## 1. Class Distribution

In [ ]:
train_data = pd.read_csv("../data/processed/train.csv")
test_data = pd.read_csv("../data/processed/test.csv")

label_col = "Label" if "Label" in train_data.columns else "label"

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Train distribution
train_counts = train_data[label_col].value_counts()
train_counts.plot(kind="bar", ax=axes[0], color=sns.color_palette("Set2", len(train_counts)))
axes[0].set_title("Training Set — Class Distribution")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=45)

# Test distribution
test_counts = test_data[label_col].value_counts()
test_counts.plot(kind="bar", ax=axes[1], color=sns.color_palette("Set2", len(test_counts)))
axes[1].set_title("Test Set — Class Distribution")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 2. Anomaly Score Distribution

In [ ]:
iso_model = joblib.load("../models/isolation_forest.pkl")
scaler = joblib.load("../models/scaler.pkl")
le = joblib.load("../models/label_encoder.pkl")

feature_cols = [c for c in test_data.columns if c not in ["Label", "label"]]
X_test = scaler.transform(test_data[feature_cols].values)
y_test = test_data[label_col].values

anomaly_scores = -iso_model.decision_function(X_test)
is_attack = y_test != "BENIGN"

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Overall
axes[0].hist(anomaly_scores, bins=100, edgecolor="black", alpha=0.7, color="steelblue")
axes[0].set_title("Overall Anomaly Score Distribution")
axes[0].set_xlabel("Anomaly Score")
axes[0].set_ylabel("Frequency")

# Split by class
axes[1].hist(anomaly_scores[~is_attack], bins=80, alpha=0.6, label="BENIGN", density=True, color="green")
axes[1].hist(anomaly_scores[is_attack], bins=80, alpha=0.6, label="Attack", density=True, color="red")
axes[1].set_title("Score Distribution by Class")
axes[1].set_xlabel("Anomaly Score")
axes[1].legend()

# Box plot by top classes
top_classes = pd.Series(y_test).value_counts().head(6).index.tolist()
box_data = [anomaly_scores[y_test == c] for c in top_classes]
axes[2].boxplot(box_data, labels=top_classes, patch_artist=True,
                boxprops=dict(facecolor="lightblue"))
axes[2].set_title("Anomaly Score by Class (Top 6)")
axes[2].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## 3. Feature Importance

In [ ]:
rf_model = joblib.load("../models/random_forest.pkl")

importances = rf_model.feature_importances_
feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=True)
top_20 = feat_imp.tail(20)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(top_20)))
top_20.plot(kind="barh", ax=ax, color=colors, edgecolor="black", linewidth=0.5)
ax.set_title("Top 20 Feature Importances (Random Forest)")
ax.set_xlabel("Importance")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.show()

## 4. Confusion Matrix Heatmap

In [ ]:
y_true = le.transform(y_test)
y_pred = rf_model.predict(X_test)
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="YlOrRd",
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_title("Confusion Matrix Heatmap")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. ROC Curves

In [ ]:
n_classes = len(le.classes_)
y_true_bin = label_binarize(y_true, classes=range(n_classes))
rf_proba = rf_model.predict_proba(X_test)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.tab10(np.linspace(0, 1, n_classes))

for i in range(n_classes):
    if y_true_bin[:, i].sum() == 0:
        continue
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], rf_proba[:, i])
    roc_auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=colors[i], linewidth=2,
            label=f"{le.classes_[i]} (AUC = {roc_auc_val:.3f})")

ax.plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.5)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — One-vs-Rest")
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Performance Comparison

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Random Forest metrics
rf_acc = accuracy_score(y_true, y_pred)
rf_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
rf_prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
rf_rec = recall_score(y_true, y_pred, average="weighted", zero_division=0)

# Isolation Forest metrics (binary: attack vs benign)
binary_true = (y_true != le.transform(["BENIGN"])[0]).astype(int)
binary_pred = (iso_model.predict(X_test) == -1).astype(int)
iso_acc = accuracy_score(binary_true, binary_pred)
iso_f1 = f1_score(binary_true, binary_pred, zero_division=0)
iso_prec = precision_score(binary_true, binary_pred, zero_division=0)
iso_rec = recall_score(binary_true, binary_pred, zero_division=0)

metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score"],
    "Isolation Forest": [iso_acc, iso_prec, iso_rec, iso_f1],
    "Random Forest": [rf_acc, rf_prec, rf_rec, rf_f1],
})

x_pos = np.arange(len(metrics_df))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x_pos - width/2, metrics_df["Isolation Forest"], width,
               label="Isolation Forest", color="coral", edgecolor="black")
bars2 = ax.bar(x_pos + width/2, metrics_df["Random Forest"], width,
               label="Random Forest", color="steelblue", edgecolor="black")

ax.set_xticks(x_pos)
ax.set_xticklabels(metrics_df["Metric"])
ax.set_ylabel("Score")
ax.set_title("Model Performance Comparison")
ax.legend()
ax.set_ylim(0, 1.15)
ax.grid(axis="y", alpha=0.3)

# Value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{bar.get_height():.3f}", ha="center", fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{bar.get_height():.3f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## 7. Detection Timeline

In [ ]:
# Simulated detection timeline over a 24-hour period
np.random.seed(42)
n_events = 200
base_time = datetime(2026, 1, 15, 0, 0, 0)
timestamps = [base_time + timedelta(seconds=np.random.randint(0, 86400)) for _ in range(n_events)]
timestamps.sort()

# Simulate severity
severities = np.random.choice(["CRITICAL", "HIGH", "MEDIUM", "LOW"], size=n_events, p=[0.1, 0.3, 0.4, 0.2])
scores = np.random.exponential(scale=0.5, size=n_events) + 0.2

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Timeline scatter
color_map = {"CRITICAL": "red", "HIGH": "orange", "MEDIUM": "gold", "LOW": "green"}
for sev in ["LOW", "MEDIUM", "HIGH", "CRITICAL"]:
    mask = severities == sev
    axes[0].scatter(
        [timestamps[i] for i in range(n_events) if mask[i]],
        scores[mask], c=color_map[sev], label=sev, alpha=0.7, s=30, edgecolors="black", linewidth=0.3
    )
axes[0].set_title("Detection Timeline (24h)")
axes[0].set_ylabel("Anomaly Score")
axes[0].legend(loc="upper right")

# Hourly counts
hours = [t.hour for t in timestamps]
hourly_counts = pd.Series(hours).value_counts().sort_index()
axes[1].bar(hourly_counts.index, hourly_counts.values, color="steelblue", edgecolor="black")
axes[1].set_title("Detections per Hour")
axes[1].set_xlabel("Hour of Day")
axes[1].set_ylabel("Count")
axes[1].set_xticks(range(24))

plt.tight_layout()
plt.show()

## 8. Severity Distribution

In [ ]:
severity_counts = pd.Series(severities).value_counts()
colors_pie = ["red", "orange", "gold", "green"]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pie chart
axes[0].pie(
    severity_counts.values, labels=severity_counts.index,
    colors=colors_pie, autopct="%1.1f%%", startangle=90,
    wedgeprops=dict(edgecolor="black", linewidth=0.5)
)
axes[0].set_title("Alert Severity Distribution")

# Bar chart
severity_counts.plot(kind="bar", ax=axes[1], color=colors_pie, edgecolor="black")
axes[1].set_title("Alert Counts by Severity")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

## 9. Summary Dashboard

In [ ]:
print("\n" + "=" * 65)
print("              VISUALIZATION DASHBOARD — SUMMARY")
print("=" * 65)

print(f"""
  DATA OVERVIEW
  ────────────────────────────────────────
  Training samples:   {len(train_data):>10,}
  Test samples:       {len(test_data):>10,}
  Features:           {len(feature_cols):>10}
  Attack classes:     {len(le.classes_):>10}

  MODEL PERFORMANCE
  ────────────────────────────────────────
  Random Forest Accuracy:   {rf_acc:.4f}
  Random Forest F1:         {rf_f1:.4f}
  Isolation Forest F1:      {iso_f1:.4f}

  ALERT SUMMARY
  ────────────────────────────────────────
  Total alerts generated:   {n_events:>10}
  Critical:                 {severity_counts.get('CRITICAL', 0):>10}
  High:                     {severity_counts.get('HIGH', 0):>10}
  Medium:                   {severity_counts.get('MEDIUM', 0):>10}
  Low:                      {severity_counts.get('LOW', 0):>10}

  TOP 5 FEATURES
  ────────────────────────────────────────""")

for feat in feat_imp.tail(5).index[::-1]:
    print(f"    {feat:<40s} {feat_imp[feat]:.4f}")

print("\n" + "=" * 65)